In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.metrics import classification_report

In [ ]:
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [3]:
class CustomDataset(Dataset):
    def __init__(self, inputs, targets, device):
        self.inputs = torch.tensor(inputs, dtype=torch.float32).to(device)
        self.targets = torch.tensor(targets, dtype=torch.int64).to(device)

    def __len__(self):
        return len(self.inputs)
    
    def __getitem__(self, idx):
        return self.inputs[idx], self.targets[idx]

class NeuralModel(nn.Module):
    def __init__(self, input_shape):
        super().__init__()
        self.layer1 = nn.Linear(input_shape, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 32)
        self.layer4 = nn.Linear(32, 2)

    def forward(self, x):
        x = F.relu(self.layer1(x))
        x = F.relu(self.layer2(x))
        x = F.relu(self.layer3(x))
        x = self.layer4(x)
        return F.log_softmax(x, dim=1)

In [ ]:
df_linis_crowd = pd.read_csv('df_linis_crowd.csv')

X = df_linis_crowd.drop(['Unnamed: 0', 'ttext', 'ttype'], axis=1).values
y = df_linis_crowd['ttype'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

trainset = torch.utils.data.DataLoader(CustomDataset(X_train, y_train, device), batch_size=128, shuffle=True)
testset = torch.utils.data.DataLoader(CustomDataset(X_test, y_test, device), batch_size=128, shuffle=False)
model = NeuralModel(input_shape=X_train.shape[1]).to(device)

loss_function = nn.NLLLoss()
optimizer = optim.Adam(model.parameters(), lr=0.00001)

for epoch in range(20):
    for data in trainset:
        X, y = data
        model.zero_grad()
        output = model(X.float())
        loss = loss_function(output, y.long())
        loss.backward()
        optimizer.step()

y_pred = []
with torch.no_grad():
    for data in testset:
        X, y = data
        output = model(X.float())
        predictions = torch.argmax(output, dim=1).cpu().numpy()
        y_pred.extend(predictions)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.55      0.75      0.63     33577
           1       0.61      0.39      0.48     34474

    accuracy                           0.57     68051
   macro avg       0.58      0.57      0.56     68051
weighted avg       0.58      0.57      0.55     68051



In [5]:
df_morphy = pd.read_csv('df_morphy.csv')

X = df_morphy.drop(['Unnamed: 0', 'ttext', 'ttype'], axis=1).values
y = df_morphy['ttype'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

trainset = torch.utils.data.DataLoader(CustomDataset(X_train, y_train, device), batch_size=128, shuffle=True)
testset = torch.utils.data.DataLoader(CustomDataset(X_test, y_test, device), batch_size=128, shuffle=False)
model = NeuralModel(input_shape=X_train.shape[1]).to(device)

loss_function = nn.NLLLoss()
optimizer = optim.Adam(model.parameters(), lr=0.00001)

for epoch in range(20):
    for data in trainset:
        X, y = data
        model.zero_grad()
        output = model(X.float())
        loss = loss_function(output, y.long())
        loss.backward()
        optimizer.step()

y_pred = []
with torch.no_grad():
    for data in testset:
        X, y = data
        output = model(X.float())
        predictions = torch.argmax(output, dim=1).cpu().numpy()
        y_pred.extend(predictions)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.60      0.54      0.57     33577
           1       0.59      0.65      0.62     34474

    accuracy                           0.60     68051
   macro avg       0.60      0.60      0.59     68051
weighted avg       0.60      0.60      0.59     68051



In [ ]:
df_tfidf = pd.read_csv('df_tfidf.csv')

X = df_tfidf.drop(['Unnamed: 0', 'ttext', 'ttype'], axis=1).values
y = df_tfidf['ttype'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

trainset = torch.utils.data.DataLoader(CustomDataset(X_train, y_train, device), batch_size=128, shuffle=True)
testset = torch.utils.data.DataLoader(CustomDataset(X_test, y_test, device), batch_size=128, shuffle=False)
model = NeuralModel(input_shape=X_train.shape[1]).to(device)

loss_function = nn.NLLLoss()
optimizer = optim.Adam(model.parameters(), lr=0.00001)

for epoch in range(20):
    for data in trainset:
        X, y = data
        model.zero_grad()
        output = model(X.float())
        loss = loss_function(output, y.long())
        loss.backward()
        optimizer.step()

y_pred = []
with torch.no_grad():
    for data in testset:
        X, y = data
        output = model(X.float())
        predictions = torch.argmax(output, dim=1).cpu().numpy()
        y_pred.extend(predictions)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.70      0.67      0.68     33577
           1       0.69      0.72      0.70     34474

    accuracy                           0.69     68051
   macro avg       0.69      0.69      0.69     68051
weighted avg       0.69      0.69      0.69     68051



In [7]:
df_my_word2vec = pd.read_csv('df_my_word2vec.csv')

X = df_my_word2vec.drop(['Unnamed: 0', 'ttext', 'ttype'], axis=1).values
y = df_my_word2vec['ttype'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

trainset = torch.utils.data.DataLoader(CustomDataset(X_train, y_train, device), batch_size=128, shuffle=True)
testset = torch.utils.data.DataLoader(CustomDataset(X_test, y_test, device), batch_size=128, shuffle=False)
model = NeuralModel(input_shape=X_train.shape[1]).to(device)

loss_function = nn.NLLLoss()
optimizer = optim.Adam(model.parameters(), lr=0.00001)

for epoch in range(20):
    for data in trainset:
        X, y = data
        model.zero_grad()
        output = model(X.float())
        loss = loss_function(output, y.long())
        loss.backward()
        optimizer.step()

y_pred = []
with torch.no_grad():
    for data in testset:
        X, y = data
        output = model(X.float())
        predictions = torch.argmax(output, dim=1).cpu().numpy()
        y_pred.extend(predictions)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.67      0.59      0.63     33577
           1       0.64      0.72      0.68     34474

    accuracy                           0.66     68051
   macro avg       0.66      0.65      0.65     68051
weighted avg       0.66      0.66      0.65     68051



In [8]:
df_word2vec = pd.read_csv('df_word2vec.csv')

X = df_word2vec.drop(['Unnamed: 0', 'ttext', 'ttype'], axis=1).values
y = df_word2vec['ttype'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

trainset = torch.utils.data.DataLoader(CustomDataset(X_train, y_train, device), batch_size=128, shuffle=True)
testset = torch.utils.data.DataLoader(CustomDataset(X_test, y_test, device), batch_size=128, shuffle=False)
model = NeuralModel(input_shape=X_train.shape[1]).to(device)

loss_function = nn.NLLLoss()
optimizer = optim.Adam(model.parameters(), lr=0.00001)

for epoch in range(20):
    for data in trainset:
        X, y = data
        model.zero_grad()
        output = model(X.float())
        loss = loss_function(output, y.long())
        loss.backward()
        optimizer.step()

y_pred = []
with torch.no_grad():
    for data in testset:
        X, y = data
        output = model(X.float())
        predictions = torch.argmax(output, dim=1).cpu().numpy()
        y_pred.extend(predictions)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.66      0.64      0.65     33577
           1       0.66      0.69      0.67     34474

    accuracy                           0.66     68051
   macro avg       0.66      0.66      0.66     68051
weighted avg       0.66      0.66      0.66     68051



In [9]:
df_my_fastText = pd.read_csv('df_my_fastText.csv')

X = df_my_fastText.drop(['Unnamed: 0', 'ttext', 'ttype'], axis=1).values
y = df_my_fastText['ttype'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

trainset = torch.utils.data.DataLoader(CustomDataset(X_train, y_train, device), batch_size=128, shuffle=True)
testset = torch.utils.data.DataLoader(CustomDataset(X_test, y_test, device), batch_size=128, shuffle=False)
model = NeuralModel(input_shape=X_train.shape[1]).to(device)

loss_function = nn.NLLLoss()
optimizer = optim.Adam(model.parameters(), lr=0.00001)

for epoch in range(20):
    for data in trainset:
        X, y = data
        model.zero_grad()
        output = model(X.float())
        loss = loss_function(output, y.long())
        loss.backward()
        optimizer.step()

y_pred = []
with torch.no_grad():
    for data in testset:
        X, y = data
        output = model(X.float())
        predictions = torch.argmax(output, dim=1).cpu().numpy()
        y_pred.extend(predictions)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.65      0.64      0.64     33577
           1       0.65      0.67      0.66     34474

    accuracy                           0.65     68051
   macro avg       0.65      0.65      0.65     68051
weighted avg       0.65      0.65      0.65     68051



In [10]:
df_my_fastText = pd.read_csv('df_fastText.csv')

X = df_my_fastText.drop(['Unnamed: 0', 'ttext', 'ttype'], axis=1).values
y = df_my_fastText['ttype'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

trainset = torch.utils.data.DataLoader(CustomDataset(X_train, y_train, device), batch_size=128, shuffle=True)
testset = torch.utils.data.DataLoader(CustomDataset(X_test, y_test, device), batch_size=128, shuffle=False)
model = NeuralModel(input_shape=X_train.shape[1]).to(device)

loss_function = nn.NLLLoss()
optimizer = optim.Adam(model.parameters(), lr=0.00001)

for epoch in range(20):
    for data in trainset:
        X, y = data
        model.zero_grad()
        output = model(X.float())
        loss = loss_function(output, y.long())
        loss.backward()
        optimizer.step()

y_pred = []
with torch.no_grad():
    for data in testset:
        X, y = data
        output = model(X.float())
        predictions = torch.argmax(output, dim=1).cpu().numpy()
        y_pred.extend(predictions)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.71      0.67      0.69     33577
           1       0.70      0.74      0.72     34474

    accuracy                           0.70     68051
   macro avg       0.70      0.70      0.70     68051
weighted avg       0.70      0.70      0.70     68051

